# Causal Trace Recipe

This recipe shows the clean-versus-corrupt activation-patching pattern behind a causal trace. We use a tiny local transformer-like model so the notebook runs without network downloads. The goal is to ask: when a corrupted prompt changes the output, which activation sites and token positions recover the clean output when patched back in?

The recipe uses public TorchLens APIs only: `tl.log_forward_pass`, `tl.sites`, `find_sites`, `fork().attach_hooks().replay()`, and `tl.viz.causal_trace_heatmap`.


In [ ]:
from __future__ import annotations

from collections.abc import Callable
from importlib import util
from pathlib import Path
from types import ModuleType
from typing import Any

import torch

import torchlens as tl
import torchlens.viz  # attaches the public viz submodule as tl.viz


def load_shared_helpers() -> ModuleType:
    """Load helpers from notebooks/total_audit/_shared.py.

    Returns
    -------
    ModuleType
        Imported shared-helper module.
    """

    helper_path = Path.cwd() / "notebooks" / "total_audit" / "_shared.py"
    spec = util.spec_from_file_location("torchlens_total_audit_shared", helper_path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Could not load shared helper module from {helper_path}")
    module = util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


shared = load_shared_helpers()
model = shared.tiny_transformer(seed=63)
clean_tokens = torch.tensor([[1, 2, 3, 4]])
corrupt_tokens = torch.tensor([[1, 7, 3, 4]])

First capture the clean and corrupted forward passes with `intervention_ready=True`. That keeps enough replay metadata for the later activation patches.


In [ ]:
clean_log = tl.log_forward_pass(model, clean_tokens, vis_opt="none", intervention_ready=True)
corrupt_log = tl.log_forward_pass(model, corrupt_tokens, vis_opt="none", intervention_ready=True)

clean_logits = clean_log.layer_list[-1].activation.detach()
corrupt_logits = corrupt_log.layer_list[-1].activation.detach()


def output_divergence(candidate_logits: torch.Tensor) -> float:
    """Measure last-token divergence from the clean output.

    Parameters
    ----------
    candidate_logits:
        Candidate logits from a replayed or captured run.

    Returns
    -------
    float
        L2 distance from the clean last-position logits.
    """

    delta = candidate_logits[:, -1, :] - clean_logits[:, -1, :]
    return float(torch.linalg.vector_norm(delta).detach())


corrupt_divergence = output_divergence(corrupt_logits)
print(f"Corrupt output divergence: {corrupt_divergence:.4f}")
assert corrupt_divergence > 0.0

A causal-trace sweep has two axes here: TorchLens sites and token positions. `tl.sites` gives a structured, repeatable way to name the operation family we want to sweep. We resolve those selectors against the corrupted log, then patch one position at a time from the clean log.


In [ ]:
def patch_position(source_activation: torch.Tensor, position: int) -> Callable[..., torch.Tensor]:
    """Create a hook that patches one sequence position from a clean activation.

    Parameters
    ----------
    source_activation:
        Clean activation tensor from the same site.
    position:
        Sequence position to patch.

    Returns
    -------
    Callable[..., torch.Tensor]
        TorchLens hook callable.
    """

    def _hook(activation: torch.Tensor, *, hook: Any) -> torch.Tensor:
        """Replace one position while leaving the rest of the activation intact."""

        patched = activation.clone()
        patched[:, position, :] = source_activation[:, position, :].to(
            device=activation.device,
            dtype=activation.dtype,
        )
        return patched

    return _hook


site_collection = tl.sites("relu", ops="relu", modes=("clean_patch",))
site_labels: list[str] = []
for site_spec in site_collection:
    site_labels.extend(corrupt_log.find_sites(site_spec.selector, max_fanout=20).labels())
site_labels = sorted(set(site_labels))
print(site_labels)
assert site_labels

In [ ]:
scores = torch.zeros(len(site_labels), clean_tokens.shape[1])
for site_index, label in enumerate(site_labels):
    clean_activation = clean_log.find_sites(tl.label(label)).first().activation.detach()
    for position in range(clean_tokens.shape[1]):
        patched = corrupt_log.fork(f"patch_{label}_pos_{position}")
        patched.attach_hooks(tl.label(label), patch_position(clean_activation, position)).replay()
        patched_logits = patched.layer_list[-1].activation.detach()
        scores[site_index, position] = corrupt_divergence - output_divergence(patched_logits)

print(scores)
assert torch.isfinite(scores).all()

Positive scores mean the patch moved the corrupted output closer to the clean output. The public visualization helper accepts any 2D score array, so the same call works for larger site-by-token sweeps.


In [ ]:
ax = tl.viz.causal_trace_heatmap(scores, signs="all", cmap="viridis")
ax.set_xlabel("token position")
ax.set_ylabel("site")
ax.set_yticks(range(len(site_labels)), site_labels)
ax.set_title("Clean-patch rescue score")